In [5]:
import sympy as sp

x = sp.symbols('x')
tau1 = sp.Function('tau_1')(x)
tau2 = sp.Function('tau_2')(x)

# Now derivatives work symbolically:
sp.diff(tau1, x)   # → Derivative(tau_1(x), x)


Derivative(tau_1(x), x)

#
From this 12d metric, one immediately finds that the nonvanishing components of the Riemann tensor are $\hat{\Gamma}^p_{mn} = \Gamma^p{}_{mn}$, $\hat{\Gamma}^a{}_{bm} = \frac{1}{2}g^{ac}g_{bc,m}$, $\hat{\Gamma}^m{}_{ab} = - \frac{1}{2}g^{mp}g_{ab,p}$.
And we evaluate that $a$ for rows and $b$ for columns, we have

\begin{equation}
\Gamma^a{}_{bm}
=\frac{1}{2 \tau_2^2}\begin{pmatrix}
    -\tfrac12 \nabla_m|\tau|^2 & |\tau|^2\nabla_m\tau_1 - \tau_1 \nabla_m|\tau|^2 \\
    \nabla_m\tau_1 & \tfrac12 \nabla_m|\tau|^2
\end{pmatrix}
\end{equation}

\begin{equation}
\Gamma_{abm}
= \frac{1}{2\tau_2}\begin{pmatrix}
    -\nabla_m\ln\tau_2
    & \nabla_m\tau_1 - \tau_1\nabla_m\ln\tau_2 \\
    \nabla_m\tau_1 - \tau_1\nabla_m\ln\tau_2 & \nabla_m|\tau|^2 - |\tau|^2\nabla_m\ln\tau_2
\end{pmatrix}
=\frac{1}{2}g_{ab,m}
\end{equation}

\begin{equation}
\Gamma^m{}_{ab} = -\frac{1}{2\tau_2}\begin{pmatrix}
    -\nabla^m\ln\tau_2 & \nabla^m\tau_1 - \tau_1\nabla^m\ln\tau_2 \\
    \nabla^m\tau_1-\tau_1\nabla^m\ln\tau_2 & \nabla^m|\tau|^2 - |\tau|^2\nabla^m\ln\tau_2
\end{pmatrix}
\end{equation}

---
Here $|\tau|^2 = \tau_1^2 + \tau_2^2$.
In our script, we don't care about the $m$ subscript, it suffices to replace all $\nabla_m$ with a simple derivative with respect to x.

In [27]:

# Shorthands
tau_sq = tau1**2 + tau2**2                # |tau|^2
dtau1 = sp.diff(tau1, x)                  # nabla tau_1
dtau_sq = sp.diff(tau_sq, x)              # nabla |tau|^2
dln_tau2 = sp.diff(sp.log(tau2), x)       # nabla ln(tau_2)

# Metric
M = sp.Matrix([[1/tau2, tau1/tau2], [tau1/tau2, tau_sq/tau2]])

# Gamma^a_{bm} — upper a, lower b  (rows = a, cols = b)
Gamma_up_a = sp.Rational(1,2) / tau2**2 * sp.Matrix([
    [-sp.Rational(1,2)*dtau_sq,  tau_sq*dtau1 - tau1*dtau_sq],
    [dtau1,                      sp.Rational(1,2)*dtau_sq     ]
])

# Gamma_{abm} = (1/2) g_{ab,m}
Gamma_low = sp.simplify(M * Gamma_up_a)

# Gamma^m_{ab}  (replacing nabla^m -> diff w.r.t. x, same expressions)
Gamma_up_m = -sp.Rational(1,2) / tau2 * sp.Matrix([
    [-dln_tau2,                    dtau1 - tau1*dln_tau2          ],
    [dtau1 - tau1*dln_tau2,        dtau_sq - tau_sq*dln_tau2      ]
])


In [22]:
Riemann_ambn = -M * sp.diff(Gamma_up_a, x) - M * Gamma_up_a* Gamma_up_a

In [40]:
sp.simplify(Riemann_ambn)

Matrix([
[                                                                                                                                     (2*tau_2(x)*Derivative(tau_2(x), (x, 2)) + Derivative(tau_1(x), x)**2 - 3*Derivative(tau_2(x), x)**2)/(4*tau_2(x)**3),                                                                                                                                                             (2*tau_1(x)*tau_2(x)*Derivative(tau_2(x), (x, 2)) + tau_1(x)*Derivative(tau_1(x), x)**2 - 3*tau_1(x)*Derivative(tau_2(x), x)**2 - 2*tau_2(x)**2*Derivative(tau_1(x), (x, 2)) + 4*tau_2(x)*Derivative(tau_1(x), x)*Derivative(tau_2(x), x))/(4*tau_2(x)**3)],
[(2*tau_1(x)*tau_2(x)*Derivative(tau_2(x), (x, 2)) + tau_1(x)*Derivative(tau_1(x), x)**2 - 3*tau_1(x)*Derivative(tau_2(x), x)**2 - 2*tau_2(x)**2*Derivative(tau_1(x), (x, 2)) + 4*tau_2(x)*Derivative(tau_1(x), x)*Derivative(tau_2(x), x))/(4*tau_2(x)**3), (2*tau_1(x)**2*tau_2(x)*Derivative(tau_2(x), (x, 2)) + tau_1(x)**2*Derivativ

In [ ]:
Riemann_mnab = M * (Gamma_up_a * Gamma_up_a)
Riemann_mnab = sp.simplify(Riemann_mnab)
# all derivatives are understood to be covariant derivatives
# this needs to be followed by antisymmetrization
Riemann_mnab

Matrix([
[         (Derivative(tau_1(x), x)**2 + Derivative(tau_2(x), x)**2)/(4*tau_2(x)**3),                                                                                                  (Derivative(tau_1(x), x)**2 + Derivative(tau_2(x), x)**2)*tau_1(x)/(4*tau_2(x)**3)],
[(Derivative(tau_1(x), x)**2 + Derivative(tau_2(x), x)**2)*tau_1(x)/(4*tau_2(x)**3), (tau_1(x)**2*Derivative(tau_1(x), x)**2 + tau_1(x)**2*Derivative(tau_2(x), x)**2 + tau_2(x)**2*Derivative(tau_1(x), x)**2 + tau_2(x)**2*Derivative(tau_2(x), x)**2)/(4*tau_2(x)**3)]])

In [39]:
A = Gamma_up_m[0,0] * Gamma_up_a[0,1]+Gamma_up_m[0,1] * Gamma_up_a[1,1]
sp.simplify(A)

(-Derivative(tau_1(x), x)**2 - Derivative(tau_2(x), x)**2)*tau_1(x)/(4*tau_2(x)**3)

# $R_{mn12}$
We find
$$
R_{mn12} = \frac{1}{2\tau_2^2}[\nabla_m\tau_2\nabla_n\tau_1-\nabla_m\tau_1\nabla_n\tau_2]
$$

# $R_{abab}$
One can then go on to compute that
$$
R_{1212} = \frac{1}{4\tau_2^2}|\nabla \tau|^2
$$
which does not have any spacetime indices.